## 1. 필요한 라이브러리 불러오기

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import time
from collections import OrderedDict

## 2. 하이퍼 파라미터 및 gpu 사용

In [ ]:
# Hyper Parameters
IMAGE_SIZE = 224
LEARNING_RATE = 0.1
EPOCHS = 20
BATCH_SIZE = 64

# GPU
device = ("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(device)

2.9.0+cu126
cuda


## 3. 데이터 준비

In [ ]:
# 데이터 준비
cifar_mean = [0.4914, 0.4822, 0.4465]
cifar_std = [0.2023, 0.1994, 0.2010]

# Standard Data Augmentation (Translation + Mirroring)
train_transform = transforms.Compose([
    # 1. Translation
    transforms.RandomCrop(32, padding=4),
    # 2. Mirroring
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])

train_data = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

test_data = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

train_dataloader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_dataloader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:04<00:00, 41.0MB/s]


In [ ]:
# 데이터 형상 확인
images, labels = next(iter(train_dataloader))
print(images[0].shape)

torch.Size([3, 32, 32])


## 4. Block 단위 클래스

In [ ]:
# Dense Block (Bottleneck 1x1Conv-3x3Conv)
class DenseBlock(nn.Module):
    def __init__(self, num_input_features, growth_rate, bn_size, drop_rate):
        super(DenseBlock, self).__init__()
        self.drop_rate = drop_rate

        # 1x1 Conv: 4*k(growth rate)
        self.conv1 = nn.Sequential(
            nn.BatchNorm2d(num_input_features),
            nn.ReLU(inplace=True),
            nn.Conv2d(num_input_features, bn_size * growth_rate, kernel_size=1, stride=1, bias=False)
        )
        # 3x3 Conv: k features
        self.conv2 = nn.Sequential(
            nn.BatchNorm2d(bn_size * growth_rate),
            nn.ReLU(inplace=True),
            nn.Conv2d(bn_size * growth_rate, growth_rate, kernel_size=3, stride=1, padding=1, bias=False)
        )
        
    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        
        # Dropout
        if self.drop_rate > 0:
            out = F.dropout(out, p=self.drop_rate, training=self.training)

        # concatenation
        return torch.cat([x, out], 1)

# Transition Block
class TransitionBlock(nn.Module):
    def __init__(self, num_input_features, num_output_features):
        super(TransitionBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.BatchNorm2d(num_input_features),
            nn.ReLU(inplace=True),
            nn.Conv2d(num_input_features, num_output_features, kernel_size=1, stride=1, bias=False),
            nn.AvgPool2d(kernel_size=2, stride=2)
        )

    def forward(self, x):
        out = self.conv(x)
        return out

## 5. DenseNet 클래스

In [ ]:
class DenseNet(nn.Module):
    def __init__(self, growth_rate=32, block_config=(6, 12, 24, 16),
                 num_init_features=64, bn_size=4, drop_rate=0,
                 num_classes=10, compression=0.5):
        super(DenseNet, self).__init__()
        
        self.features = nn.Sequential(OrderedDict([
            ('conv0', nn.Conv2d(3, num_init_features, kernel_size=3, stride=1, padding=1, bias=False)),
            ('norm0', nn.BatchNorm2d(num_init_features)),
            ('relu0', nn.ReLU(inplace=True)),
        ]))

        num_features = num_init_features
        
        # growth_rate에 따른 DenseBlock
        for i, num_layers in enumerate(block_config):
            block = nn.Sequential()
            for j in range(num_layers):
                layer = DenseBlock(num_features, growth_rate, bn_size, drop_rate)
                block.add_module(f'denselayer{j+1}', layer)
                num_features += growth_rate

            self.features.add_module(f'denseblock{i+1}', block)

            # Transition Block
            if i != len(block_config) - 1:
                num_output_features = int(num_features * compression)
                trans = TransitionBlock(num_features, num_output_features)
                self.features.add_module(f'transition{i+1}', trans)
                num_features = num_output_features

        self.final_norm = nn.BatchNorm2d(num_features)
        self.classifier = nn.Linear(num_features, num_classes)

        self._init_weights()

    def forward(self, x):
        features = self.features(x)
        out = F.relu(self.final_norm(features), inplace=True) 
        out = F.adaptive_avg_pool2d(out, (1, 1))
        out = torch.flatten(out, 1)
        out = self.classifier(out)
        return out

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.constant_(m.bias, 0)

## 6. 훈련 준비 및 optimizer 선언

In [ ]:
# 훈련 준비
model = DenseNet(growth_rate=32, block_config=[6, 12, 24, 16], num_classes=10)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=1e-4)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[150, 225], gamma=0.1)

## 7. 훈련

In [ ]:
from google.colab import drive
import os
import time
import torch

drive.mount('/content/drive')
save_dir = '/content/drive/{원하는 경로}'
os.makedirs(save_dir, exist_ok=True)

print("학습 시작")
best_acc = 0.0

for epoch in range(EPOCHS):
    start_time = time.time()

    model.train()
    running_loss = 0.0

    for inputs, labels in train_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in test_dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_acc = 100 * correct / total
    epoch_time = time.time() - start_time

    if epoch_acc > best_acc:
        best_acc = epoch_acc
        save_path = os.path.join(save_dir, f"DenseNet121_best.pth")
        torch.save(model.state_dict(), save_path)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | "
          f"Loss: {running_loss/len(train_dataloader):.4f} | "
          f"Test Acc: {epoch_acc:.2f}% | "
          f"Time: {epoch_time:.1f}s")

print(f"학습 종료 | 최고 정확도: {best_acc:.2f}%")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
학습 시작
Epoch [1/20] Loss: 0.558 | Test Acc: 77.74% | Time: 162.7s
Epoch [2/20] Loss: 0.484 | Test Acc: 82.63% | Time: 162.8s
Epoch [3/20] Loss: 0.436 | Test Acc: 82.07% | Time: 162.7s
Epoch [4/20] Loss: 0.385 | Test Acc: 84.62% | Time: 163.0s
Epoch [5/20] Loss: 0.265 | Test Acc: 88.30% | Time: 162.7s
Epoch [6/20] Loss: 0.232 | Test Acc: 88.80% | Time: 162.5s
Epoch [7/20] Loss: 0.219 | Test Acc: 89.25% | Time: 162.3s
Epoch [8/20] Loss: 0.208 | Test Acc: 89.16% | Time: 162.5s
Epoch [9/20] Loss: 0.201 | Test Acc: 89.67% | Time: 162.2s
Epoch [10/20] Loss: 0.191 | Test Acc: 89.31% | Time: 162.3s
Epoch [11/20] Loss: 0.183 | Test Acc: 89.47% | Time: 162.7s
Epoch [12/20] Loss: 0.168 | Test Acc: 89.61% | Time: 162.4s
Epoch [13/20] Loss: 0.165 | Test Acc: 89.88% | Time: 162.8s
Epoch [14/20] Loss: 0.164 | Test Acc: 89.78% | Time: 162.6s
Epoch [15/20] Loss: 0.165 | Test A